# 01 — Data prep (SpaCy)

**Project:** SpaCy vs LLM agreement on Kafka (DE/EN)  
**Team:** Dominik Soballa, Luca Bouché  

This notebook loads the raw Kafka texts, runs SpaCy (`de_core_news_md` / `en_core_web_md`), samples **300 sentences per language** (`seed=42`), and writes token tables to `data/processed/`.

You can also run the same pipeline via:
```bash
source .venv/bin/activate
python scripts/run_data_prep.py
```

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.config import N_SENTENCES_PER_LANGUAGE, RANDOM_SEED, RAW_DE, RAW_EN
from src.spacy_pipeline import load_spacy, load_text, sample_sentence_ids, sentences_to_frame

sns.set_theme()
print("seed", RANDOM_SEED, "n_sents", N_SENTENCES_PER_LANGUAGE)

In [ ]:
def prepare_language(lang: str, raw_rel: str) -> pd.DataFrame:
    text = load_text(ROOT / raw_rel)
    nlp = load_spacy(lang)
    df = sentences_to_frame(nlp, text, language=lang)
    non_space = df.loc[~df["is_space"]]
    sizes = non_space.groupby("sent_id").size()
    usable = df[df["sent_id"].isin(sizes.loc[sizes >= 3].index)].copy()
    sent_ids = sample_sentence_ids(usable, n=N_SENTENCES_PER_LANGUAGE, seed=RANDOM_SEED)
    sample = usable[usable["sent_id"].isin(sent_ids) & ~usable["is_space"]][
        ["language", "sent_id", "tok_id", "token", "upos", "lemma"]
    ].copy()
    out = ROOT / "data" / "processed" / f"tokens_{lang}_sample.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    sample.to_csv(out, index=False)
    print(lang, "->", out.name, "rows", len(sample), "sents", sample.sent_id.nunique())
    return sample

# Prefer loading existing exports if present (fast path)
de_path = ROOT / "data/processed/tokens_de_sample.csv"
en_path = ROOT / "data/processed/tokens_en_sample.csv"
if de_path.exists() and en_path.exists():
    sample_de = pd.read_csv(de_path)
    sample_en = pd.read_csv(en_path)
    print("Loaded existing samples")
else:
    sample_de = prepare_language("de", RAW_DE)
    sample_en = prepare_language("en", RAW_EN)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, df, title in [
    (axes[0], sample_de, "DE UPOS (sample)"),
    (axes[1], sample_en, "EN UPOS (sample)"),
]:
    order = df.upos.value_counts().index
    sns.countplot(data=df, y="upos", order=order, ax=ax, color="#4C78A8")
    ax.set_title(title)
fig.tight_layout()
fig_dir = ROOT / "figures"
fig_dir.mkdir(exist_ok=True)
fig.savefig(fig_dir / "01_upos_dist_sample.png", dpi=150)
plt.show()
sample_de.upos.value_counts().head(), sample_en.upos.value_counts().head()